# 総当たり（2遺伝子の対戦）で上位の並びを精密にする

ノートブック 03（Yes/No の M3s）は全遺伝子を速く採点できますが、**上位どうしの細かい並び**は点数が接近していて不安定です。このノートブックでは、M3s で上位 `TOP_K`（既定 30）遺伝子に絞り、その中の**全ペア**について「3条件のどれかによりよく当てはまるのはどちらか」を2択で聞きます。対戦結果から **Bradley-Terry モデル**で強さを推定して並べ直します。

| 段階 | 中身 |
|---|---|
| 1. 絞り込み | M3s（Yes/No、両順）で全遺伝子を採点し、上位 `TOP_K` を選ぶ（ノートブック 03 の出力 `*_m3s.csv` があれば再利用） |
| 2. 総当たり | 上位 `TOP_K` の全ペア（30 なら 435 組）を、左右を入れ替えて2回ずつ聞く |
| 3. 強さの推定 | Bradley-Terry（MM 法）で強さ（log π）を推定。参考に Elo レーティングと勝率も出す |

- **質問は M3s と同じ3条件**です（広い聞き方は有名な遺伝子に流れることが検証で分かっているため）。2択なので「どちらがよりよく当てはまるか」と聞きます。
- 1回ごとの判断が単純な2択で、同じ遺伝子が 29 回ずつ異なる相手と比べられるため、上位の並びが安定します（ただし下の検証結果のとおり、既知遺伝子を上に置く力は M3s より下がることが多い）。
- **Bradley-Terry モデル**：遺伝子 i が j に勝つ確率を π_i / (π_i + π_j) と置き、全対戦の結果に最もよく合う π を求めます。勝つ確率（0〜1 のやわらかい勝ち数）をそのまま使えます。全勝・全敗でも発散しないよう、各遺伝子に「強さ 1 の仮想相手と 0.5 勝 0.5 敗」を足しています。

## 実機検証の結果（txgemma-9b-chat Q6_K、5疾患、M3s 上位30、435ペア × 両順、`scripts/tournament.py`）

| 疾患 | AUC 既知 vs その他（上位30内）：M3s の並び | 総当たり（BT） | 差 | 1番が選ばれる確率 | 安定性（ペア2分割の順位相関） |
|---|---|---|---|---|---|
| 軟骨無形成症 | **0.596** | 0.490 | −0.106 | 0.86 | 0.92 |
| 関節リウマチ | **0.835** | 0.744 | −0.091 | 0.76 | 0.95 |
| 前立腺がん | 0.549 | **0.688** | **+0.139** | 0.87 | 0.98 |
| 統合失調症 | **0.831** | 0.656 | −0.175 | 0.97 | 0.79 |
| シスチン尿症 | **0.821** | 0.786 | −0.035 | 0.99 | 0.93 |
| **平均** | **0.726** | 0.673 | −0.054 | 0.89 | 0.91 |

**結論：総当たりは並びを安定させますが、既知遺伝子を上に置く力は平均すると M3s の並びより下がりました。** 使うなら「M3s とは別の角度の並び」として参照してください。
- **並びの安定性は高い**（ペアを無作為に2分割して推定した順位の相関 0.79〜0.98）。上位の細かい並びは再現します。
- **2つを直接比べると、病気の中心経路に最も近い遺伝子が勝ち続けます。** 軟骨無形成症では FGFR3 経路の候補（IHH・MAP2K1・FRS2）、関節リウマチでは炎症性サイトカイン（IL17A・IL23A）、統合失調症ではグルタミン酸受容体（GRIN1・GRIN2A）が上位に来て、別の仕組みで効く既知標的（GH1・JAK・DRD2）が下がりました。
- **前立腺がんだけは総当たりの方が良かった**（GNRHR 19 → 4 位、CYP17A1 22 → 9 位）。既知標的がホルモン経路という中心経路にまとまっているためと考えられます。
- **2択では「1番」を選ぶ癖が非常に強い**（平均 0.89、シスチン尿症では 0.99）。両順の平均で打ち消していますが、癖が強いほど残る差が小さくなり、判断の情報が薄くなります（統合失調症・シスチン尿症では勝率がほぼ 0.5 に集まった）。


## 注意
- **2択では「先に書かれた方」を選びやすい癖が強く出ます**（1番が勝つ確率の平均が 0.76〜0.99）。全ペアを左右入れ替えて2回聞き、平均することで打ち消しています（左右の片方だけで推定すると、並びが逆転するほど偏ります）。
- 選択肢の記号は数字（1 / 2）を使います。A / B にすると、このモデルでは答えの位置に A・B のトークンがほとんど出てこず、確率が読めませんでした。
- 対戦数は TOP_K² に比例して増えます（30 → 870 回、GGUF で約15分）。
- 質問文・前置き・推定の手順は `scripts/tournament.py` と同じです（変えたら検証し直してください）。
- モデルが無い環境ではモック（擬似乱数）で動きます。数値に意味はありません。

### このセルがすること：準備

In [1]:
import os, re, math, glob, json, time, random, hashlib, itertools, urllib.request
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
ROOT = os.path.abspath("..")
print("ready:", ROOT)

ready: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02


## 疾患の選択

### このセルがすること：`data/diseases.json` の登録疾患から1つ選び、病名・症状の箇条書き・遺伝子リストを読み込む
- ノートブック 03 と同じ読み込み方です。

In [2]:
DISEASE_KEY = "ra"                    # "ra" / "scz" / "cystinuria" / "prostate_cancer" / "achondroplasia"
GENE_SET = "set100"                   # "set100" / "set1000" / "known" / "candidates"

REGISTRY = json.load(open(os.path.join(ROOT, "data", "diseases.json"), encoding="utf-8"))
print("登録疾患:", {k: v["name"] for k, v in REGISTRY.items()})
D = REGISTRY[DISEASE_KEY]
DISEASE = D["name"]
DISEASE_INFO = D["info"][:5]
GENE_FILE = os.path.join(ROOT, "data", "genes", f"{D['gene_prefix']}_{GENE_SET}.tsv")
print("選択:", DISEASE, "| 遺伝子リスト:", os.path.basename(GENE_FILE))

登録疾患: {'ra': 'rheumatoid arthritis', 'scz': 'schizophrenia', 'cystinuria': 'cystinuria', 'prostate_cancer': 'prostate cancer', 'achondroplasia': 'achondroplasia'}
選択: rheumatoid arthritis | 遺伝子リスト: ra_set100.tsv


## 設定

### このセルがすること：絞り込む数・M3s の結果の再利用・モデルの選択を宣言する
- `TOP_K`：総当たりにかける上位遺伝子の数（対戦数は TOP_K × (TOP_K − 1)。30 で 870 回）。
- `REUSE_M3S`：ノートブック 03 の出力 `outputs/<セット名>_m3s.csv` があれば、段階1（M3s の採点）を省いてそれを使う。

In [3]:
TOP_K = 30
REUSE_M3S = True
N_CTX = 2048

# --- モデルの選択（ノートブック 03 と同じ仕組み） ---
BACKEND = "auto"                                   # "auto" / "gguf" / "ollama" / "mock"
MODEL_DIR = os.path.expanduser("~/llm/models")
OLLAMA_URL = "http://localhost:11434"
MODEL_SELECT = "auto"                              # "auto" / 一覧の番号 / 名前の一部
PREFER = ("txgemma", "medgemma", "gemma")
models = []
for h in sorted(glob.glob(os.path.join(MODEL_DIR, "**", "*.gguf"), recursive=True)):
    models.append({"kind": "gguf", "name": os.path.basename(h), "path": h, "size_gb": round(os.path.getsize(h) / 1e9, 2)})
try:
    with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=3) as r:
        for m in json.load(r).get("models", []):
            models.append({"kind": "ollama", "name": m["name"], "path": m["name"], "size_gb": round(m.get("size", 0) / 1e9, 2)})
except Exception:
    pass
chosen = None
if BACKEND != "mock":
    if isinstance(MODEL_SELECT, int):
        chosen = models[MODEL_SELECT]
    else:
        pool = [m for m in models if BACKEND == "auto" or m["kind"] == BACKEND]
        if MODEL_SELECT != "auto": pool = [m for m in pool if MODEL_SELECT.lower() in m["name"].lower()]
        ranked = sorted(pool, key=lambda m: (min([i for i, p in enumerate(PREFER) if p in m["name"].lower()] or [99]),
                                             0 if m["kind"] == "gguf" else 1, m["name"]))
        chosen = ranked[0] if ranked else None
USE_LLM = chosen is not None
BACKEND_USED = chosen["kind"] if chosen else "mock"
MODEL_PATH = chosen["path"] if chosen else None
print("使えるモデル:"); [print(f"  [{i}] {m['kind']:6s} {m['size_gb']:6.2f} GB  {m['name']}") for i, m in enumerate(models)]
print("選択:", f"{BACKEND_USED}: {MODEL_PATH}" if USE_LLM else "モック（擬似乱数。数値に意味なし）")
OUT_DIR = os.path.join(ROOT, "outputs"); os.makedirs(OUT_DIR, exist_ok=True)
SET_NAME = os.path.basename(GENE_FILE).replace(".tsv", "")

使えるモデル:
  [0] gguf     7.59 GB  txgemma-9b-chat-Q6_K.gguf
  [1] ollama   7.59 GB  txgemma-9b-chat-q6_k:latest
  [2] ollama  17.40 GB  gemma3:27b
  [3] ollama   4.68 GB  qwen2.5:7b
  [4] ollama   1.16 GB  bge-m3:latest
  [5] ollama   9.28 GB  qwen3:14b
  [6] ollama   4.37 GB  cniongolo/biomistral:latest
  [7] ollama   4.01 GB  gemma3:4b-it-qat
  [8] ollama   5.23 GB  deepseek-r1:8b
  [9] ollama   5.23 GB  qwen3:8b
  [10] ollama   8.99 GB  qwen2.5:14b
  [11] ollama   4.92 GB  llama3.1:latest
選択: gguf: /Users/yoshinorisatomi/llm/models/txgemma-9b-chat-Q6_K.gguf


## 入力遺伝子

### このセルがすること：遺伝子リストを読み、プロンプトに入れる名前（記号＋タンパク質名）を作る

In [4]:
genes = pd.read_csv(GENE_FILE, sep="\t", dtype=str).fillna("")
for col in ("protein_name_uniprot", "gene_name", "category", "label"):
    if col not in genes.columns: genes[col] = ""
genes["gene_label"] = [f"{g['symbol']} ({(g['protein_name_uniprot'] or g['gene_name']).split('|')[0]})" if (g["protein_name_uniprot"] or g["gene_name"]) else g["symbol"] for _, g in genes.iterrows()]
genes = genes.set_index("symbol", drop=False)
print(len(genes), "genes")

100 genes


## 質問とプロンプトの組み立て

### このセルがすること：段階1（Yes/No の M3s）と段階2（2択）の質問・前置きを定義する
- **M3s**（段階1）：ノートブック 03 と同じ文面・同じ前置き。
- **2択**（段階2）：同じ3条件を「番号付きの2遺伝子のどちらがよりよく当てはまるか」で聞く。末尾は「**Answer: **」（空白まで）で、この直後の数字 1 / 2 の確率を読む。

In [5]:
M3S_QUESTION = "Consider these three criteria:\n(a) Inhibiting or activating {gene} could plausibly treat {disease} or improve at least one of the symptoms listed above.\n(b) Even outside the causal pathway, modulating {gene} could counteract the abnormal process described above through a parallel or opposing pathway.\n(c) {gene}'s own substrate, ligand, pathway, cell type or circuit precisely matches the mechanism of {disease} described above (a similar but distinct role, even in the same gene family, does not count).\nDoes this gene meet at least one of these criteria?"
YN_NOTE = 'Note: the vast majority of human genes are NOT drug targets for any given disease. Answer Yes only when there is clear evidence or a clear mechanistic link; otherwise answer No. Membership in the same gene family, superfamily or protein class as a true disease gene (e.g. being another member of the same transporter, channel, receptor or enzyme family) is NOT sufficient evidence by itself. Judge each gene on whether ITS OWN specific substrate, ligand, cargo or interaction partner matches the mechanism described above, not on family resemblance alone.\n'
PAIR_QUESTION = "Consider these three criteria:\n(a) Inhibiting or activating the gene could plausibly treat {disease} or improve at least one of the symptoms listed above.\n(b) Even outside the causal pathway, modulating the gene could counteract the abnormal process described above through a parallel or opposing pathway.\n(c) The gene's own substrate, ligand, pathway, cell type or circuit precisely matches the mechanism of {disease} described above (a similar but distinct role, even in the same gene family, does not count).\nWhich of the two numbered genes above better meets at least one of these criteria?"
PAIR_NOTE = 'Note: the vast majority of human genes are NOT drug targets for any given disease. Pick a gene only when there is a clear, specific reason; membership in the same gene family as a known disease gene is NOT sufficient by itself.\n'
ROLE = 'You are an expert in drug discovery and human disease biology.\n'

def bullets(): return "\n".join(f"- {b}" for b in DISEASE_INFO)

def yn_prefix(order):
    options = "Answer each question with Yes or No." if order == "yes_first" else "Answer each question with No or Yes."
    return (ROLE + YN_NOTE + f"Disease: {DISEASE}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{bullets()}\n{options}\n\n")

def yn_parts(label, symbol):   # 遺伝子行と質問は別々にトークン化する（ノートブック 03・検証スクリプトと同じ）
    return [f"Gene: {label}\n", f"M3s. {M3S_QUESTION.format(disease=DISEASE, gene=symbol)} Answer:"]

def pair_prefix():
    return (ROLE + "You will be shown two candidate genes for one disease, and asked which ONE number is the better answer.\n" + PAIR_NOTE +
            f"Disease: {DISEASE}\nTarget symptoms and the organ, cell and functional abnormalities behind them:\n{bullets()}\n"
            "Answer with a single number, 1 or 2, only. No words, no explanation.\n\n")

def pair_text(label1, label2):
    return f"Candidate genes:\n1. {label1}\n2. {label2}\n" + f"Question: {PAIR_QUESTION.format(disease=DISEASE)} Answer: "

print(pair_prefix() + pair_text("TNF (tumor necrosis factor)", "ACTB (actin beta)"))

You are an expert in drug discovery and human disease biology.
You will be shown two candidate genes for one disease, and asked which ONE number is the better answer.
Note: the vast majority of human genes are NOT drug targets for any given disease. Pick a gene only when there is a clear, specific reason; membership in the same gene family as a known disease gene is NOT sufficient by itself.
Disease: rheumatoid arthritis
Target symptoms and the organ, cell and functional abnormalities behind them:
- Joint pain, swelling and morning stiffness, symmetric in the small joints of hands and feet: caused by chronic inflammation of the synovial membrane that lines the joints
- Progressive joint deformity and loss of function: invasive growth of the synovial lining cells and excessive bone resorption by bone-degrading cells erode cartilage and bone
- The inflammation is sustained by the adaptive immune system: self-reactive T cells, antibody-producing B cells and infiltrating macrophages accumu

## エンジン（GGUF を llama-cpp-python で直接動かす）

### `yes_prob(label, symbol, order)` — M3s の Yes の確率（段階1）
### `pair_prob(label1, label2)` — 2択で「1番」が選ばれる確率（段階2）
どちらも、保存しておいた前置きの状態を（コンテキストを空にしてから）復元し、遺伝子の部分と質問だけを処理して、答えの位置の確率を読みます。
Yes/No は綴り違い（Yes / yes / YES …）を合算し、2択は数字 1 と 2 のトークンだけで正規化します。

### このセルがすること：エンジンを読み込み、前置き3通り（Yes/No の両順、2択）を処理して保存する

In [6]:
def softmax(l):
    l = np.asarray(l, float); p = np.exp(l - l.max()); return p / p.sum()

if BACKEND_USED == "gguf":
    from llama_cpp import Llama
    llm = Llama(model_path=MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=-1, logits_all=False, verbose=False)
    def tok(text, bos=False): return llm.tokenize(text.encode("utf-8"), add_bos=bos, special=bos)
    def ids_of(spellings):
        out = []
        for s in spellings:
            t = tok(s)
            if len(t) == 1 and t[0] not in out: out.append(t[0])
        return out
    YES_IDS = ids_of(["Yes", " Yes", "yes", " yes", "YES", " YES"]); NO_IDS = ids_of(["No", " No", "no", " no", "NO", " NO"])
    DIGIT = {n: tok(str(n)) for n in (1, 2)}
    assert all(len(v) == 1 for v in DIGIT.values()), f"数字が1トークンになっていません: {DIGIT}"
    STATE = {}
    for name, text in (("yes_first", yn_prefix("yes_first")), ("no_first", yn_prefix("no_first")), ("pair", pair_prefix())):
        llm.reset(); llm.eval(tok(text, bos=True)); STATE[name] = llm.save_state()
    def logits_after(state, texts):
        llm.reset(); llm.load_state(STATE[state])
        for t in ([texts] if isinstance(texts, str) else texts): llm.eval(tok(t))
        return np.ctypeslib.as_array(llm._ctx.get_logits(), shape=(llm.n_vocab(),)).astype(np.float64)
    def lse(v): m = max(v); return m + math.log(sum(math.exp(x - m) for x in v))
    def yes_prob(label, symbol, order):
        lg = logits_after(order, yn_parts(label, symbol)); lg = lg - lg.max(); lp = lg - math.log(np.exp(lg).sum())
        ly, ln = lse([lp[i] for i in YES_IDS]), lse([lp[i] for i in NO_IDS])
        return 1 / (1 + math.exp(ln - ly))
    def pair_prob(label1, label2):
        lg = logits_after("pair", pair_text(label1, label2))
        return float(softmax([lg[DIGIT[1][0]], lg[DIGIT[2][0]]])[0])
    print("GGUF engine ready")
elif BACKEND_USED == "ollama":
    def ollama_generate(prompt, **opt):
        req = urllib.request.Request(OLLAMA_URL + "/api/generate", headers={"Content-Type": "application/json"},
                                     data=json.dumps({"model": MODEL_PATH, "prompt": prompt, "raw": True, "stream": False, "logprobs": True,
                                                      "top_logprobs": 20, "options": {"temperature": 0, "num_predict": 1, **opt}}).encode())
        with urllib.request.urlopen(req, timeout=300) as r: out = json.load(r)
        lps = out.get("logprobs") or []
        return {t["token"]: t["logprob"] for t in (lps[0].get("top_logprobs", []) if lps else [])}
    def wrap(body, tail):   # Gemma 系のテンプレート（他のモデルはノートブック 03 の ollama_template を参照）
        return f"<start_of_turn>user\n{body}<end_of_turn>\n<start_of_turn>model\n{tail}"
    def pick(top, keys):
        floor = min(top.values()) if top else -20
        return [max([v for t, v in top.items() if t.strip().lower() == k.lower()] or [floor]) for k in keys]
    def yes_prob(label, symbol, order):
        body = yn_prefix(order) + "".join(yn_parts(label, symbol)).replace(" Answer:", "")
        ly, ln = pick(ollama_generate(wrap(body, "Answer:")), ["yes", "no"]); return 1 / (1 + math.exp(ln - ly))
    def pair_prob(label1, label2):
        body = pair_prefix() + pair_text(label1, label2).replace(" Answer: ", "")
        return float(softmax(pick(ollama_generate(wrap(body, "Answer: ")), ["1", "2"]))[0])
    print("Ollama を使います（前置きを毎回処理するので遅い経路です）:", MODEL_PATH)
else:
    def yes_prob(label, symbol, order): return int(hashlib.md5((label + order).encode()).hexdigest(), 16) % 1000 / 1000
    def pair_prob(label1, label2): return int(hashlib.md5((label1 + "|" + label2).encode()).hexdigest(), 16) % 1000 / 1000
    print("警告: モデルが無いのでモック（擬似乱数）です。")

t0 = time.time()
print("test pair TNF vs ACTB:", round(pair_prob("TNF (tumor necrosis factor)", "ACTB (actin beta)"), 3),
      "| ACTB vs TNF:", round(pair_prob("ACTB (actin beta)", "TNF (tumor necrosis factor)"), 3), f"| {time.time() - t0:.2f}s")

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


GGUF engine ready


test pair TNF vs ACTB: 1.0 | ACTB vs TNF: 0.184 | 2.43s


## 段階1：M3s で全遺伝子を採点し、上位 `TOP_K` に絞る

### このセルがすること：ノートブック 03 の出力があれば読み込み、無ければ M3s を両順で聞いて対数オッズの平均を出す

In [7]:
logit = lambda x: math.log(max(x, 1e-6) / max(1 - x, 1e-6))
m3s_csv = os.path.join(OUT_DIR, f"{SET_NAME}_m3s.csv")
if REUSE_M3S and os.path.exists(m3s_csv):
    m3s = pd.read_csv(m3s_csv).set_index("symbol")["logodds"]
    print("ノートブック 03 の結果を再利用:", m3s_csv)
else:
    t0 = time.time()
    m3s = pd.Series({s: np.mean([logit(yes_prob(genes.at[s, "gene_label"], s, o)) for o in ("yes_first", "no_first")]) for s in genes.index})
    print(f"M3s を採点しました（{time.time() - t0:.0f}s）")
genes["m3s_logodds"] = m3s.reindex(genes.index)
top = genes.sort_values("m3s_logodds", ascending=False).head(TOP_K).index.tolist()
print(f"上位 {TOP_K}:", top)
print("そのうち既知:", [s for s in top if genes.at[s, "category"] == "known"])
print("上位に入らなかった既知:", [s for s in genes.index[genes.category == "known"] if s not in top])

ノートブック 03 の結果を再利用: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_m3s.csv
上位 30: ['TNF', 'IL6', 'IL1B', 'IL1R1', 'JAK1', 'TNFRSF1A', 'IL6R', 'CD40', 'JAK2', 'IL17RA', 'IL17A', 'JAK3', 'IL18', 'HLA-DRB1', 'TLR4', 'IL21', 'STAT4', 'TNFRSF14', 'IL10', 'CD40LG', 'IL23A', 'MMP3', 'TNFRSF9', 'IL2RA', 'PTGS2', 'MMP1', 'CSF2', 'IL15', 'NFKB1', 'IL6ST']
そのうち既知: ['TNF', 'IL6', 'IL1R1', 'JAK1', 'IL6R', 'JAK2', 'JAK3', 'PTGS2']
上位に入らなかった既知: ['TNFSF11', 'CD80', 'DHFR', 'DHODH', 'CD86', 'MS4A1', 'NR3C1']


## 段階2：総当たり

### このセルがすること：上位 `TOP_K` の全ペアを、左右を入れ替えて2回ずつ聞き、CSV に書く
- `p_a_first`：a を1番に置いたとき a が選ばれる確率。`p_a_second`：a を2番に置いたとき a が選ばれる確率。

In [8]:
rows, t0 = [], time.time()
pairs = list(itertools.combinations(range(len(top)), 2))
for k, (a, b) in enumerate(pairs):
    la, lb = genes.at[top[a], "gene_label"], genes.at[top[b], "gene_label"]
    rows.append({"a": top[a], "b": top[b], "p_a_first": pair_prob(la, lb), "p_a_second": 1 - pair_prob(lb, la)})
    if (k + 1) % 100 == 0 or k + 1 == len(pairs): print(f"{k + 1}/{len(pairs)} pairs ({time.time() - t0:.0f}s)")
pr = pd.DataFrame(rows)
pr.to_csv(os.path.join(OUT_DIR, f"{SET_NAME}_tournament_pairs.csv"), index=False)
first_pos = float(np.mean(list(pr["p_a_first"]) + list(1 - pr["p_a_second"])))
print(f"1番に置かれた方が選ばれる確率の平均 = {first_pos:.3f}（0.5 なら位置の癖なし。両順の平均で打ち消す）")

100/435 pairs (208s)


200/435 pairs (422s)


300/435 pairs (635s)


400/435 pairs (871s)


435/435 pairs (951s)
1番に置かれた方が選ばれる確率の平均 = 0.756（0.5 なら位置の癖なし。両順の平均で打ち消す）


## 段階3：強さの推定

### `bradley_terry(n, games)` — Bradley-Terry モデル（MM 法）
どんな def か：games は (i, j, w) の並び（w = i が j に勝つ確率、1試合ぶん）。各遺伝子の強さ π を、勝つ確率が π_i / (π_i + π_j) になるように反復で推定します。各遺伝子に「強さ 1 の仮想相手との 0.5 勝 0.5 敗」を足して、全勝・全敗でも発散しないようにしています。
return：log π（numpy 配列、幾何平均が 1 になるよう正規化）。

### `elo(n, games)` — Elo レーティング（参考）
どんな def か：初期 1500 から1試合ごとに更新します。対戦の順番で結果が変わるので、順番をシャッフルして 50 回平均します。

### このセルがすること：強さを推定し、安定性（ペアを無作為に2分割して推定した並びの一致度）を出し、CSV に書く

In [9]:
def bradley_terry(n, games, iters=500, prior=0.5):
    """games: [(i, j, w)]（w = i が j に勝つ確率、1試合ぶん）。MM 法で強さ π を推定し log π を返す。
    prior：各遺伝子に「強さ1の仮想相手」との prior 勝ち・prior 負けを足して、全勝・全敗でも発散しないようにする。"""
    W = np.full(n, prior)                                          # やわらかい勝ち数（仮想相手からの prior 勝ちを含む）
    pairs = {}
    for i, j, w in games:
        W[i] += w; W[j] += 1 - w
        pairs[(i, j)] = pairs.get((i, j), 0) + 1
    pi = np.ones(n)
    for _ in range(iters):
        denom = 2 * prior / (pi + 1)                               # 仮想相手（π=1）との 2*prior 試合
        for (i, j), c in pairs.items():
            d = c / (pi[i] + pi[j]); denom[i] += d; denom[j] += d
        new = W / denom
        new /= math.exp(np.mean(np.log(new)))                      # 幾何平均を 1 に固定
        if np.max(np.abs(np.log(new) - np.log(pi))) < 1e-9: pi = new; break
        pi = new
    return np.log(pi)


def elo(n, games, k=24.0, shuffles=50, seed=0):
    """Elo レーティング（初期 1500）。対戦の順番で結果が変わるので、順番をシャッフルして平均する。"""
    rng, total = random.Random(seed), np.zeros(n)
    for _ in range(shuffles):
        r, g = np.full(n, 1500.0), games[:]; rng.shuffle(g)
        for i, j, w in g:
            e = 1 / (1 + 10 ** ((r[j] - r[i]) / 400))
            r[i] += k * (w - e); r[j] -= k * (w - e)
        total += r
    return total / shuffles


def spearman(a, b): return float(pd.Series(a).rank().corr(pd.Series(b).rank()))

idx = {s: i for i, s in enumerate(top)}
games = [g for r in pr.itertuples() for g in ((idx[r.a], idx[r.b], r.p_a_first), (idx[r.a], idx[r.b], r.p_a_second))]
n = len(top)
bt, el = bradley_terry(n, games), elo(n, games)
win, cnt = np.zeros(n), np.zeros(n)
for i, j, w in games: win[i] += w; win[j] += 1 - w; cnt[i] += 1; cnt[j] += 1
half = np.random.default_rng(0).permutation(len(pr)) % 2
g_h = [[g for h, r in zip(half, pr.itertuples()) if h == k for g in ((idx[r.a], idx[r.b], r.p_a_first), (idx[r.a], idx[r.b], r.p_a_second))] for k in (0, 1)]
stability = spearman(bradley_terry(n, g_h[0]), bradley_terry(n, g_h[1]))

res = pd.DataFrame({"symbol": top, "gene_label": [genes.at[s, "gene_label"] for s in top], "category": [genes.at[s, "category"] for s in top],
                    "m3s_logodds": [genes.at[s, "m3s_logodds"] for s in top], "bt": bt, "elo": el, "win_rate": win / cnt})
res["m3s_rank"] = np.arange(1, n + 1)
res["bt_rank"] = res["bt"].rank(ascending=False, method="min").astype(int)
res = res.sort_values("bt_rank").reset_index(drop=True)
res.round(4).to_csv(os.path.join(OUT_DIR, f"{SET_NAME}_tournament.csv"), index=False)
print(f"安定性（ペアを2分割して推定した BT の順位相関）= {stability:.3f} | BT と Elo の順位相関 = {spearman(bt, el):.3f} | BT と M3s = {spearman(bt, res.sort_values('m3s_rank')['m3s_logodds']):.3f}")
res[["bt_rank", "m3s_rank", "symbol", "category", "bt", "elo", "win_rate", "m3s_logodds"]].round(3)

安定性（ペアを2分割して推定した BT の順位相関）= 0.948 | BT と Elo の順位相関 = 1.000 | BT と M3s = 0.516


,bt_rank,m3s_rank,symbol,category,bt,elo,win_rate,m3s_logodds
0,1,2,IL6,known,1.594,1704.215,0.811,7.832
1,2,1,TNF,known,1.197,1656.269,0.745,10.207
2,3,3,IL1B,candidate,1.085,1641.958,0.724,7.318
3,4,11,IL17A,candidate,0.983,1632.618,0.704,5.081
4,5,6,TNFRSF1A,candidate,0.905,1621.565,0.689,6.309
5,6,4,IL1R1,known,0.900,1618.088,0.688,6.750
6,7,21,IL23A,candidate,0.880,1617.652,0.684,4.010
7,8,10,IL17RA,candidate,0.874,1613.720,0.683,5.275
8,9,16,IL21,candidate,0.793,1604.449,0.667,4.461
9,10,20,CD40LG,candidate,0.697,1595.973,0.647,4.099


## 評価

### このセルがすること：上位 `TOP_K` の中で、M3s の並びと総当たりの並びのどちらが既知遺伝子を上に置けたかを比べる
- `AUC known vs others`（上位 `TOP_K` の中だけ）：既知を候補・ダミーより上に置けるか。
- `既知の平均順位` と `上位10件中の既知の数` も出します。

In [10]:
def auc(pos, neg):
    pos, neg = np.asarray(list(pos), float), np.asarray(list(neg), float)
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    return float(((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg)))
if not USE_LLM: print("警告: モックの数値です。")
kn = res["category"].eq("known")
ev = pd.DataFrame({name: {"AUC known vs others（上位内）": auc(res.loc[kn, col], res.loc[~kn, col]),
                          "既知の平均順位": res.loc[kn, rk].mean(), "上位10件中の既知": int((res[rk] <= 10)[kn].sum())}
                   for name, col, rk in (("M3s（段階1）", "m3s_logodds", "m3s_rank"), ("Bradley-Terry（総当たり）", "bt", "bt_rank"))}).round(3)
display(ev)
print("既知遺伝子の順位（M3s → 総当たり）:")
res.loc[kn, ["symbol", "m3s_rank", "bt_rank", "bt", "win_rate"]].round(3)

,M3s（段階1）,Bradley-Terry（総当たり）
AUC known vs others（上位内）,0.835,0.744
既知の平均順位,8.125,10.125
上位10件中の既知,6.000,3.000


既知遺伝子の順位（M3s → 総当たり）:


,symbol,m3s_rank,bt_rank,bt,win_rate
0,IL6,2,1,1.594,0.811
1,TNF,1,2,1.197,0.745
5,IL1R1,4,6,0.900,0.688
10,IL6R,7,11,0.629,0.632
12,JAK2,9,13,0.535,0.613
13,JAK1,5,14,0.406,0.585
14,PTGS2,25,15,0.007,0.497
18,JAK3,12,19,-0.340,0.422


## 対話型グラフ（ホバーで遺伝子名）

### このセルがすること：強さの棒グラフ、M3s の順位 → 総当たりの順位の比較、対戦の勝率ヒートマップを描く
- 同じ図を `outputs/<セット名>_tournament_charts.html` に保存します。plotly.js を HTML に埋め込むので、ネット接続なしで開けます。

In [11]:
import plotly.graph_objects as go
COL = {"known": "#2a78d6", "candidate": "#eb6834", "random": "#1baf7a"}

fig = go.Figure(go.Bar(x=res["symbol"], y=res["bt"], marker_color=[COL.get(c, "#888") for c in res["category"]],
                       customdata=res[["gene_label", "category", "m3s_rank", "win_rate"]].values,
                       hovertemplate="<b>%{x}</b><br>%{customdata[0]}<br>%{customdata[1]}<br>強さ log π %{y:.2f}<br>M3s の順位 %{customdata[2]}"
                                     "<br>勝率 %{customdata[3]:.2f}<extra></extra>"))
fig.update_layout(title=f"{DISEASE}: Bradley-Terry strength (top {TOP_K} by M3s)  — 青＝既知、橙＝候補、緑＝ダミー",
                  yaxis_title="強さ log π", template="plotly_white", width=1100, height=420, xaxis_tickangle=-60)
fig.show()

fig2 = go.Figure()
for _, r in res.iterrows():
    fig2.add_trace(go.Scatter(x=[0, 1], y=[r["m3s_rank"], r["bt_rank"]], mode="lines+markers+text", text=[r["symbol"], r["symbol"]],
                              textposition=["middle left", "middle right"], line=dict(color=COL.get(r["category"], "#888"), width=2 if r["category"] == "known" else 1),
                              marker=dict(size=7), showlegend=False, hovertemplate=f"<b>{r['symbol']}</b>（{r['category']}）<br>順位 %{{y}}<extra></extra>"))
fig2.update_layout(title="順位の入れ替わり：左＝M3s（段階1）、右＝総当たり（Bradley-Terry）", template="plotly_white", width=700, height=900,
                   xaxis=dict(tickvals=[0, 1], ticktext=["M3s", "総当たり"], range=[-0.4, 1.4]), yaxis=dict(autorange="reversed", title="順位"))
fig2.show()

order = res["symbol"].tolist(); pos = {s: i for i, s in enumerate(order)}
M = np.full((n, n), np.nan)
for r in pr.itertuples():
    w = (r.p_a_first + r.p_a_second) / 2; M[pos[r.a], pos[r.b]] = w; M[pos[r.b], pos[r.a]] = 1 - w
fig3 = go.Figure(go.Heatmap(z=M, x=order, y=order, zmin=0, zmax=1, colorscale="RdBu", colorbar=dict(title="行が勝つ確率"),
                            hovertemplate="%{y} が %{x} に勝つ確率 %{z:.2f}<extra></extra>"))
fig3.update_layout(title="対戦の勝率（両順の平均）。強さの順に並べてある", template="plotly_white", width=850, height=800,
                   yaxis=dict(autorange="reversed"), xaxis_tickangle=-60)
fig3.show()

html_path = os.path.join(OUT_DIR, f"{SET_NAME}_tournament_charts.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'></head><body>")
    for i, fg in enumerate((fig, fig2, fig3)):
        f.write(fg.to_html(full_html=False, include_plotlyjs=True if i == 0 else False))
    f.write("</body></html>")
print("saved:", html_path)

saved: /Users/yoshinorisatomi/Documents/claude/gene_disease_prediction02/outputs/ra_set100_tournament_charts.html
